# Verify `training-models`' sanity checks

The gates in `scripts/sanity_check.py` need PyTorch, which this repository
deliberately does not depend on — installing it to test a measurement helper
would cost more than it buys. So CI skips those tests and this notebook runs
them where torch already exists.

It asserts nothing of its own. The assertions live in
`tests/test_sanity_check.py`, which is also what CI runs when torch happens to
be present — one source of truth, two places to run it.

**Runtime → Run all.** Green means every gate both opens on a healthy model and
shuts on a broken one; a gate that only ever reports `pass` is not a gate.


## 1. Get the skill

Colab has torch preinstalled. Only the skill itself has to arrive.


In [ ]:
import pathlib
import subprocess
import sys

import torch

REPO = pathlib.Path("skills")
if not REPO.exists():
    subprocess.run(
        ["git", "clone", "--depth", "1", "https://github.com/ImWenyaoT/skills.git", str(REPO)],
        check=True,
    )

SKILL = REPO / "skills" / "training-models"
assert (SKILL / "scripts" / "sanity_check.py").is_file(), SKILL
print("torch", torch.__version__)
print("skill at", SKILL.resolve())

## 2. Run the gate tests

Each gate is tested in both directions: a model that should pass it, and one
broken on purpose that must fail it.


In [ ]:
result = subprocess.run(
    [
        sys.executable,
        "-m",
        "unittest",
        "discover",
        "-s",
        str(SKILL / "tests"),
        "-p",
        "test_*.py",
        "-v",
    ],
    capture_output=True,
    text=True,
)
print(result.stdout[-4000:])
print(result.stderr[-4000:])
assert result.returncode == 0, "a gate did not behave as documented"
print("\nAll gates verified.")

## 3. See a report on a real model

The tests prove the gates discriminate. This shows what you actually read when
you run the checks on your own model — replace the toy below with yours.


In [ ]:
import importlib.util

import torch.nn as nn

spec = importlib.util.spec_from_file_location("sanity_check", SKILL / "scripts" / "sanity_check.py")
sc = importlib.util.module_from_spec(spec)
sys.modules["sanity_check"] = sc
spec.loader.exec_module(sc)

torch.manual_seed(0)
model = nn.Sequential(nn.Linear(6, 32), nn.ReLU(), nn.Linear(32, 4))
xb, yb = torch.randn(8, 6), torch.randint(0, 4, (8,))

sc.run_sanity_checks(model, xb, yb, nn.CrossEntropyLoss())

## 4. Watch a gate shut

A report you have only ever seen say PASS teaches you nothing about what a
failure looks like. This breaks the logits contract — a softmax before a loss
that applies its own — which is the silent one, because the loss still falls.


In [ ]:
broken = nn.Sequential(nn.Linear(6, 32), nn.ReLU(), nn.Linear(32, 4), nn.Softmax(dim=-1))
print(sc.check_logits_contract(broken, xb, nn.CrossEntropyLoss()))